C independent ENGINEERING_TEST: SETUP and zero-call sentinel only; complete notebooks remain unchecked.

In [1]:
from pathlib import Path
import sys, json, tempfile, shutil
from html import escape
from IPython.display import display, Image, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'task1').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from task1.workflow.io import read_json, digest, object_hash
from task1.workflow.g2_provider import ExperimentProvider
model_sentinel = {'calls': 0}
def reject_model_call(*args, **kwargs):
    model_sentinel['calls'] += 1
    raise RuntimeError('RECOMPUTE_FORBIDS_NEW_MODEL_CALLS')
ExperimentProvider.call = reject_model_call
EV = ROOT/'task1/evidence/goal2'
current = read_json(EV/'current_runs.json')
contract = read_json(ROOT/'task1/config/goal2/contract.json')
split = read_json(EV/'data/split_manifest.json')
MODE = 'RECOMPUTE'
ENABLE_LIVE = False
assert MODE == 'RECOMPUTE' and ENABLE_LIVE is False, '新的 LIVE 调用须使用独立命令并显式启用，默认 Notebook 不发起模型请求。'
WORK = Path(tempfile.mkdtemp(prefix='sc-g2-notebook-'))
def show(rows):
    rows = list(rows)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = ''.join('<th>'+escape(str(key))+'</th>' for key in columns)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(row.get(key, '')))+'</td>' for key in columns)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))
def figure(name):
    directory = Path(recomputed.get('figure_directory', WORK/'figures'))
    target = directory/(name+'.png')
    assert target.is_file(), f'复算图缺失: {target}'
    display(Image(filename=str(target), width=1050))
print('运行方式:', MODE, '| 新模型调用:', 0)
print('原始来源 datum:', contract['source_crs'], '| 单位: 工作米 / 原始相对秒')
print('当前固定分区:', {name:len(ids) for name,ids in split['splits'].items()})


运行方式: RECOMPUTE | 新模型调用: 0
原始来源 datum: UNVERIFIED | 单位: 工作米 / 原始相对秒
当前固定分区: {'PILOT_REGRESSION': 7, 'DEMO_MEMORY': 60, 'DEVELOPMENT': 120, 'G2_EVAL': 120, 'G3_RESERVED': 11079}


In [2]:
assert model_sentinel == {'calls': 0}
assert MODE == 'RECOMPUTE' and ENABLE_LIVE is False
assert 'pandas' not in globals()
try:
    ExperimentProvider().call('ENGINEERING_TEST_MUST_NEVER_REACH_PROVIDER', WORK/'forbidden', request_id='fixture', purpose='ENGINEERING_TEST')
except RuntimeError as error:
    assert str(error) == 'RECOMPUTE_FORBIDS_NEW_MODEL_CALLS'
else:
    raise AssertionError('provider sentinel did not reject')
assert model_sentinel == {'calls': 1}
assert not (WORK/'forbidden').exists()
shutil.rmtree(WORK)
print('C_SETUP_SENTINEL_VERIFIED; blocked_test_attempts=1; actual_model_calls=0')


C_SETUP_SENTINEL_VERIFIED; blocked_test_attempts=1; actual_model_calls=0
